# AERIS Exposure: Static Exposure Assessment

This notebook demonstrates a complete static-mode workflow:
1. Load subject data (ZIP codes + date ranges)
2. Run the OpenAQ-backed estimator
3. Interpret summary metrics, AQI, and completeness
4. Explore daily and long-format outputs

In [ ]:
import pandas as pd
from aeris.exposure import ExposureEstimatorOpenAQStatic

## 1. Prepare Input Data

Static mode requires a CSV (or DataFrame) with columns:
- `subject_id` — unique participant identifier
- `zip` or `address` (or `latitude`/`longitude`) — location
- `start_date`, `end_date` — observation window

In [ ]:
df_input = pd.read_csv("static_example.csv")
df_input.head()

## 2. Run the Estimator

Key parameters:
- `pollutant`: one of `pm25`, `pm10`, `so2`, `no2`, `co`, `o3`, or `all`
- `country`: ISO country code for ZIP geocoding (default `"us"`)
- `n_stations`: number of stations for IDW averaging (default 1 = nearest)
- `outlier_detection`: enable IQR-based outlier removal

In [ ]:
estimator = ExposureEstimatorOpenAQStatic(
    input_csv=df_input,  # accepts DataFrame or file path
    pollutant="pm25",
    # api_key="YOUR_OPENAQ_KEY",  # or set OPENAQ_API_KEY env var
)

# skip_confirm=True for non-interactive use
result = estimator.run(skip_confirm=True)
result

## 3. Interpret Results

The summary includes:
- **mean / median / std** — central tendency and dispersion
- **aqi / aqi_category** — EPA Air Quality Index
- **n_samples** — number of hourly measurements
- **completeness_pct** — data coverage as % of expected hours
- **station_id / station_name** — OpenAQ station used (audit trail)
- **distance_m** — distance from subject to monitoring station

In [ ]:
# Filter to well-covered subjects
if "completeness_pct" in result.columns:
    well_covered = result[result["completeness_pct"] >= 50]
    print(f"{len(well_covered)} of {len(result)} subjects have >= 50% data completeness")
    well_covered[["subject_id", "mean", "aqi", "aqi_category", "completeness_pct"]]

## 4. Daily Aggregates

Use `get_daily_data()` for day-level analysis.

In [ ]:
daily = estimator.get_daily_data()
if not daily.empty:
    daily.head(10)
else:
    print("No daily data available (run the estimator first).")

## 5. Long-Format (Tidy) Output

Use `get_raw_data()` for one-row-per-measurement format,
suitable for mixed-effects models in R (lme4) or statsmodels.

In [ ]:
raw = estimator.get_raw_data()
if not raw.empty:
    print(f"Shape: {raw.shape}")
    raw.head()